# 项目一：分层划分与预处理 Pipeline

目标：建立可复用的训练、验证、测试划分，以及数值/类别特征的统一预处理流程。

## 本项目的数据边界

- 训练集：约 60%，用于学习预处理参数和模型参数。
- 验证集：约 20%，用于开发阶段比较候选模型。
- 测试集：约 20%，封存到模型选择完成后再评估。

使用 `stratify=y`，使三部分的高风险学生比例尽量保持一致。

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
data_path = Path('projects/01_student_risk_prediction/data/raw/student-mat.csv')
if not data_path.exists():
    data_path = Path.home() / 'solo_work/算法工程师/projects/01_student_risk_prediction/data/raw/student-mat.csv'

df = pd.read_csv(data_path, sep=';')
y = (df['G3'] < 10).astype(int)
X = df.drop(columns=['G1', 'G2', 'G3'])

In [3]:
# 第一次划分：先留出一次性最终测试集。
X_develop, X_test, y_develop, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 第二次划分：把开发数据按 3:1 分为训练集和验证集，最终约为 60% / 20% / 20%。
X_train, X_val, y_train, y_val = train_test_split(
    X_develop, y_develop, test_size=0.25, random_state=42, stratify=y_develop
)

split_summary = pd.DataFrame({
    'samples': [len(X_train), len(X_val), len(X_test)],
    'risk_ratio': [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=['train', 'validation', 'test'])
display(split_summary)

,samples,risk_ratio
train,237,0.329114
validation,79,0.329114
test,79,0.329114


In [4]:
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(exclude='number').columns.tolist()

# 数值特征：即使当前无缺失，也保留中位数填补，使流程能处理未来数据。
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# 类别特征：用训练集中最常见类别填补；one-hot 后，未知类别不报错。
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

In [5]:
# fit_transform：只在训练集上学习填补值、均值、标准差和类别字典，然后转换训练集。
X_train_ready = preprocessor.fit_transform(X_train)

# transform：验证集和测试集只能复用训练集学到的规则。
X_val_ready = preprocessor.transform(X_val)
X_test_ready = preprocessor.transform(X_test)

print(f'原始特征数：{X_train.shape[1]}')
print(f'预处理后特征数：{X_train_ready.shape[1]}')
print(f'训练 / 验证 / 测试形状：{X_train_ready.shape} / {X_val_ready.shape} / {X_test_ready.shape}')

原始特征数：30
预处理后特征数：56
训练 / 验证 / 测试形状：(237, 56) / (79, 56) / (79, 56)


In [6]:
# 查看 one-hot 后部分特征名，理解为什么特征数从 30 增加。
feature_names = preprocessor.get_feature_names_out()
print('前 20 个预处理后特征：')
print(feature_names[:20])

前 20 个预处理后特征：
['numeric__age' 'numeric__Medu' 'numeric__Fedu' 'numeric__traveltime'
 'numeric__studytime' 'numeric__failures' 'numeric__famrel'
 'numeric__freetime' 'numeric__goout' 'numeric__Dalc' 'numeric__Walc'
 'numeric__health' 'numeric__absences' 'categorical__school_GP'
 'categorical__school_MS' 'categorical__sex_F' 'categorical__sex_M'
 'categorical__address_R' 'categorical__address_U'
 'categorical__famsize_GT3']


## 本阶段结论

预处理后特征数会增加，是因为一个类别字段会被展开成多个 0/1 特征。例如 `school` 有 `GP`、`MS` 两种取值，one-hot 后会形成对应的类别列。

下一步把 `preprocessor` 和逻辑回归封装进一个完整 Pipeline，在训练集拟合并用验证集评估风险识别能力。